In [0]:
pip install great-expectations==0.18.12


In [0]:
%restart_python

In [0]:
import great_expectations as gx

context = gx.get_context()


In [0]:
df = spark.read.table("onedevcatalog.bronze_one.customer_dim")

In [0]:
%python
context.add_datasource(
    name="my_spark_datasource",
    class_name="Datasource",
    execution_engine={
        "class_name": "SparkDFExecutionEngine"
    },
    data_connectors={
        "default_runtime_data_connector_name": {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"]
        }
    }
)

In [0]:
#datasource = context.sources.add_spark("my_spark_datasource2")

#dataframe = df

name = "my_df_asset3"

data_asset = datasource.add_dataframe_asset(name=name)

#my_batch_request = data_asset.build_batch_request(dataframe=dataframe)



In [0]:
%python


In [0]:
%python
import yaml
from great_expectations.core.expectation_suite import ExpectationSuite

suite_path = "/Workspace/Users/snehashrivastav@gmail.com/databricks-repo-fact/gx/expectations/customer_dim_suite.yaml"

with open(suite_path, "r") as f:
    suite_dict = yaml.safe_load(f)

suite_obj = ExpectationSuite(**suite_dict)

context.add_expectation_suite(
    expectation_suite=suite_obj
)

suite_name = suite_obj.expectation_suite_name
suite = context.get_expectation_suite(suite_name)

In [0]:
from great_expectations.core.batch import RuntimeBatchRequest

my_batch_request = RuntimeBatchRequest(
    datasource_name="my_spark_datasource",
    data_connector_name="default_runtime_data_connector_name",
    data_asset_name="customer_dim_temp_view",
    runtime_parameters={
        "batch_data": df
    },
    batch_identifiers={
        "default_identifier_name": "customer_dim_batch"
    }
)

validator = context.get_validator(
    batch_request=my_batch_request,
    expectation_suite=suite,
)

In [0]:
results = validator.validate()
print(results)
